In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
### 所需要的所有文件：
# 合并物流-财务-产品组-核算价-国内-用于低效-长尾 
# PLM的生命周期全表，因为涉及到停止销售时间，所以这里需要PLM的生命周期全表

In [2]:
month_date = 202510
channel = ['零售','工程','电商']
current_date = pd.Timestamp('2025-09-30')

productgroup_map={
    '吸油烟机': ['吸油烟机'],
    '灶具': ['灶具'],
    '蒸烤微合计': ['烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微'],
    '灶集成': ['灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '消毒柜': ['消毒柜'],
    '热水器': ['热水器','两用炉'],
    '净水机': ['家用净水机','商用净水机'],
    '洗碗机': ['水槽洗碗机','嵌入式洗碗机']
}

### 读取物流数据，只保留3大渠道、并且是国内的数据

In [3]:
df = pd.read_excel(fr'D:\000物料报表\{month_date}\单型号贡献-低效-长尾\合并物流-财务-产品组-核算价-国内-用于低效-长尾.xlsx')
df['物料号'] = df['商品编码'].astype(str).map(lambda x: x[:13])
print(len(df))
df.head()


356495


,商品编码,渠道,实际出库数量,物料编码,产品组,系统核算价,核算价,标准型号,国内/海外,生命周期状态,物料号
0,1009001100002,工程,1,1009001100002,蒸烤烹饪机,3550,3550,ZK50-01-F1.i,国内,停止销售,1009001100002
1,1001002100022,工程,2,1001002100022,吸油烟机,1508,3016,JC03A,国内,量产,1001002100022
2,1002003400032,工程,2,1002003400032,灶具,900,1800,TH3B,国内,退市预警,1002003400032
3,1001002000018,工程,1,1001002000018,吸油烟机,3668,3668,03-X1A,国内,量产,1001002000018
4,1003000500029,工程,1,1003000500029,消毒柜,1550,1550,ZTD100J-J31,国内,量产,1003000500029


In [7]:
# 长尾只看3大渠道。每个渠道的停止销售时间和既定时间的差距，所以只需要保留物料号、渠道、产品组、标准型号、国内/海外
df1 = df.copy()
df1 = df1[(df1['渠道'].isin(channel))&df1['国内/海外'].isin(['国内'])]
df1 = df1[['物料号','渠道','产品组','标准型号','国内/海外']].drop_duplicates().reset_index(drop=True)
df1

,物料号,渠道,产品组,标准型号,国内/海外
0,1009001100002,工程,蒸烤烹饪机,ZK50-01-F1.i,国内
1,1001002100022,工程,吸油烟机,JC03A,国内
2,1002003400032,工程,灶具,TH3B,国内
3,1001002000018,工程,吸油烟机,03-X1A,国内
4,1003000500029,工程,消毒柜,ZTD100J-J31,国内
...,...,...,...,...,...
1909,1018000500003,工程,嵌入式洗碗机,JBCD15E-W3,国内
1910,1009001300001,零售,灶蒸烤烹饪机,JZT-ZK60-01-X20Pro,国内
1911,1009000900022,工程,灶蒸烤烹饪机,JZT-ZK60-02-X3.i,国内
1912,1009000600025,工程,蒸烤烹饪机,ZK42-F2.i,国内


### 将PLM的渠道的状态信息匹配进去

In [5]:
df_product_life = pd.read_excel(fr'D:\000物料报表\{month_date}\单型号贡献-低效-长尾\产品生命周期状态全表.xlsx')
# 转换物料号列为字符串类型，并只取前13位
df_product_life[['物料号']] = df_product_life[['物料号']].astype(str).map(lambda x: x[:13])
df_product_life = df_product_life[df_product_life['物料号'].str.len()>10]
df_product_life['渠道'] = df_product_life['下属渠道']
#####
df_product_life_map_df = df_product_life[['物料号','渠道','对应渠道状态','产品状态','产品型号','停止销售时间']]
df_product_life_map_df 

,物料号,渠道,对应渠道状态,产品状态,产品型号,停止销售时间
4,1004000200090,NaN,NaN,停止发货,10T-JSG15-0606FR,NaN
5,1004000200079,NaN,NaN,停止发货,10T-JSG19-0607FR,NaN
6,1004000200084,NaN,NaN,停止发货,10T-JSG25-0608FR,NaN
7,1004000200035,NaN,NaN,停止发货,10T-JSQ16-0601,NaN
8,1004000200060,NaN,NaN,停止发货,10T-JSQ16-0601FR,NaN
...,...,...,...,...,...,...
9113,1001002200009,工程,在售,量产,iMES-45-C1,NaN
9114,1001002200005,工程,在售,量产,iMES-60-D1,NaN
9115,1001002200001,NaN,NaN,作废,iMES-C1,NaN
9116,1001002200007,工程,在售,量产,iMES-D1G-K1,NaN


### 保留各个渠道停止销售的产品

In [6]:
df_calu = pd.merge(df1,df_product_life_map_df,how='left',on=['物料号','渠道'])
df_calu['停止销售时间'] = pd.to_datetime(df_calu['停止销售时间'])
df_calu = df_calu[df_calu['渠道'].isin(channel)]
df_calu = df_calu[df_calu['对应渠道状态'].isin(['停止销售'])]
df_calu

C:\Users\zhangbon\AppData\Local\Temp\ipykernel_19484\1975798613.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_calu['停止销售时间'] = pd.to_datetime(df_calu['停止销售时间'])


,物料号,渠道,产品组,标准型号,国内/海外,对应渠道状态,产品状态,产品型号,停止销售时间
0,1009001100002,工程,蒸烤烹饪机,ZK50-01-F1.i,国内,停止销售,停止销售,ZK50-01-F1.i,2025-08-01 13:40:23
5,1018000500033,工程,嵌入式洗碗机,JBCD7E-02-V6,国内,停止销售,停止销售,JBCD7E-02-V6,2025-09-11 10:19:53
9,1001000800337,工程,吸油烟机,EH37,国内,停止销售,停止销售,CXW-258-EH37,2022-07-07 12:00:00
11,1001002000004,工程,吸油烟机,X1A,国内,停止销售,停止销售,CXW-258-X1A,2025-06-19 09:21:24
14,1009000600012,工程,蒸烤烹饪机,ZK-TS1.i,国内,停止销售,停止销售,ZK-TS1.i,2025-04-25 08:55:42
...,...,...,...,...,...,...,...,...,...
1886,1001001100034,工程,吸油烟机,EG08,国内,停止销售,停止销售,CXW-200-EG08（不带罩）,2020-08-27 12:00:00
1889,1001000500375,工程,吸油烟机,JQ31A,国内,停止销售,停止销售,CXW-358-JQ33A(不带罩）,2025-06-19 09:20:56
1895,1002001800332,零售,灶具,FD7B,国内,停止销售,停止销售,JZR-FD7B-7RⅣ,2024-08-02 17:14:08
1907,1008000300027,电商,水槽洗碗机,JBSD2T-K3AL,国内,停止销售,退市预警,JBSD2T-K3AL,2025-07-23 15:02:58


In [ ]:
from calendar import month
from pandas import DateOffset
#  只要有一个产品型号是长尾，那么这个标准型号就是长尾
for index,row in df_calu.iterrows():
    if row['渠道'] == '零售':
        if row['停止销售时间'] < current_date - pd.DateOffset(months=12):
            df_calu.loc[index,'是否长尾型号'] = '是'
    if row['渠道'] == '电商':
        if row['停止销售时间'] < current_date - pd.DateOffset(months=12):
            df_calu.loc[index,'是否长尾型号'] = '是'
    if row['渠道'] == '工程':
        if row['停止销售时间'] < current_date - pd.DateOffset(months=30):
            df_calu.loc[index,'是否长尾型号'] = '是'
df_calu['是否长尾型号'] = df_calu['是否长尾型号'].fillna('否')
df_calu

,物料号,渠道,产品组,标准型号,国内/海外,对应渠道状态,产品状态,产品型号,停止销售时间,是否长尾型号
0,1009001100002,工程,蒸烤烹饪机,ZK50-01-F1.i,国内,停止销售,停止销售,ZK50-01-F1.i,2025-08-01 13:40:23,否
5,1018000500033,工程,嵌入式洗碗机,JBCD7E-02-V6,国内,停止销售,停止销售,JBCD7E-02-V6,2025-09-11 10:19:53,否
9,1001000800337,工程,吸油烟机,EH37,国内,停止销售,停止销售,CXW-258-EH37,2022-07-07 12:00:00,是
11,1001002000004,工程,吸油烟机,X1A,国内,停止销售,停止销售,CXW-258-X1A,2025-06-19 09:21:24,否
14,1009000600012,工程,蒸烤烹饪机,ZK-TS1.i,国内,停止销售,停止销售,ZK-TS1.i,2025-04-25 08:55:42,否
...,...,...,...,...,...,...,...,...,...,...
1886,1001001100034,工程,吸油烟机,EG08,国内,停止销售,停止销售,CXW-200-EG08（不带罩）,2020-08-27 12:00:00,是
1889,1001000500375,工程,吸油烟机,JQ31A,国内,停止销售,停止销售,CXW-358-JQ33A(不带罩）,2025-06-19 09:20:56,否
1895,1002001800332,零售,灶具,FD7B,国内,停止销售,停止销售,JZR-FD7B-7RⅣ,2024-08-02 17:14:08,是
1907,1008000300027,电商,水槽洗碗机,JBSD2T-K3AL,国内,停止销售,退市预警,JBSD2T-K3AL,2025-07-23 15:02:58,否


: 

In [8]:
df_calu.to_excel(fr"C:\Users\zhangbon\Desktop\长尾明细.xlsx", index=False)


In [ ]:
df_out = pd.DataFrame()
df_out['产品类别'] = productgroup_map.keys()
for k,v in productgroup_map.items():
    vals = df_calu[(df_calu['产品组'].isin(v))&(df_calu['是否长尾型号']=='是')]['标准型号'].nunique()
    df_out.loc[df_out['产品类别']==k,'长尾标准型号数量'] = vals
    vals = df_calu[(df_calu['产品组'].isin(v))]['标准型号'].nunique()
    df_out.loc[df_out['产品类别']==k,'标准型号数量'] = vals
df_out['长尾标准型号占比'] = df_out['长尾标准型号数量']/df_out['标准型号数量']
df_out

,产品类别,零售长尾产品型号数量,零售产品型号数量,零售长尾产品型号占比,工程长尾产品型号数量,工程产品型号数量,工程长尾产品型号占比,电商长尾产品型号数量,电商产品型号数量,电商长尾产品型号占比,全渠道长尾产品型号数量,全渠道产品型号数量,全渠道长尾产品型号占比
0,吸油烟机,8.0,28.0,0.285714,25.0,46.0,0.543478,0.0,9.0,0.0,33.0,70.0,0.471429
1,灶具,18.0,35.0,0.514286,4.0,29.0,0.137931,0.0,7.0,0.0,22.0,62.0,0.354839
2,蒸烤微合计,5.0,19.0,0.263158,0.0,14.0,0.000000,0.0,17.0,0.0,5.0,31.0,0.161290
3,灶集成,2.0,14.0,0.142857,0.0,0.0,0.000000,0.0,8.0,0.0,2.0,14.0,0.142857
4,消毒柜,2.0,4.0,0.500000,0.0,1.0,0.000000,0.0,0.0,0.0,2.0,5.0,0.400000
5,热水器,7.0,7.0,1.000000,1.0,4.0,0.250000,0.0,0.0,0.0,8.0,9.0,0.888889
6,净水机,6.0,13.0,0.461538,0.0,6.0,0.000000,0.0,2.0,0.0,6.0,13.0,0.461538
7,洗碗机,4.0,24.0,0.166667,4.0,24.0,0.166667,0.0,41.0,0.0,8.0,66.0,0.121212


In [ ]:
df_out1 = pd.DataFrame()
df_out1['产品类别'] = productgroup_map.keys()
for k,v in productgroup_map.items():
    vals1 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['是否长尾型号']=='是')&(df_calu['渠道']=='零售')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'零售长尾产品型号数量'] = vals1
    vals2 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['渠道']=='零售')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'零售产品型号数量'] = vals2
    df_out1.loc[df_out1['产品类别']==k,'零售长尾产品型号占比'] = vals1/vals2 if vals2!=0 else 0
    
    vals1 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['是否长尾型号']=='是')&(df_calu['渠道']=='工程')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'工程长尾产品型号数量'] = vals1
    vals2 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['渠道']=='工程')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'工程产品型号数量'] = vals2
    df_out1.loc[df_out1['产品类别']==k,'工程长尾产品型号占比'] = vals1/vals2 if vals2!=0 else 0
    
    vals1 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['是否长尾型号']=='是')&(df_calu['渠道']=='电商')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'电商长尾产品型号数量'] = vals1
    vals2 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['渠道']=='电商')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'电商产品型号数量'] = vals2 
    df_out1.loc[df_out1['产品类别']==k,'电商长尾产品型号占比'] = vals1/vals2 if vals2!=0 else 0
    

    vals1 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['是否长尾型号']=='是')]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'全渠道长尾产品型号数量'] = vals1
    vals2 = df_calu[(df_calu['产品组'].isin(v))]['产品型号'].nunique()
    df_out1.loc[df_out1['产品类别']==k,'全渠道产品型号数量'] = vals2
    df_out1.loc[df_out1['产品类别']==k,'全渠道长尾产品型号占比'] = vals1/vals2 if vals2!=0 else 0
    
df_out1

In [10]:
with pd.ExcelWriter(fr'D:\000物料报表\{month_date}\单型号贡献-低效-长尾\长尾统计结果.xlsx') as writer:
    df_out.to_excel(writer,sheet_name='标准型号统计',index=False)
    df_out1.to_excel(writer,sheet_name='分渠道产品型号统计',index=False)
